# U-Net Seabed Segmentation with Transfer Learning

Trains a U-Net with a pretrained ResNet34 encoder on 3-channel echosounder
patches (two frequencies + their difference) for binary water/seabed
segmentation. The encoder weights come from ImageNet, and the 3-channel
input maps naturally to the pretrained architecture without any adapter layers.

In [ ]:
print("Hello, World!")

In [ ]:
from google.colab import userdata
import os

# from dotenv import load_dotenv

# # Load variables from .env
# load_dotenv()

token = os.getenv('GITHUB_TOKEN')
username = "axelfhagen"
repo = "Master"
repo_url = f"https://{token}@github.com/{username}/{repo}.git"

# Clone if not already present
if not os.path.exists('/content/Master'):
    !git clone {repo_url}
else:
    print("Master repo already cloned")

In [ ]:
!pip install -r /content/Master/code/requirements.txt

In [ ]:
import sys
from google.colab import drive

drive.mount('/content/drive')

# Add cloned repo to path so imports work
sys.path.insert(0, '/content/Master/code')

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from deep_learning.echo_dataset import (
    EchogramPatchDataset,
    get_augmentation_pipeline,
)
from deep_learning.unet_model import (
    create_unet,
    freeze_encoder,
    freeze_encoder_stages,
    unfreeze_encoder,
    get_trainable_param_count,
    get_param_groups,
    DiceBCELoss,
    train_one_epoch,
    validate,
    predict_full_echogram,
    #EarlyStopping,
)

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

In [ ]:
# --- Paths ---
DRIVE_BASE = Path('/content/drive/MyDrive/Master')
PATCH_CACHE = DRIVE_BASE / 'patch_cache'

# --- Frequencies (Hz) ---
FREQ1_HZ = 38_000
FREQ2_HZ = 200_000

# --- Patch extraction ---
PATCH_WIDTH = 256
STRIDE = 64
MAX_RANGE_M = None
MAX_RANGE_SAMPLES = None

# --- Model ---
TARGET_SIZE = (512, 512)
ENCODER_NAME = 'resnet34'
FREEZE_STRATEGY = 'encoder'
FREEZE_STAGES = 3

# --- Training ---
BATCH_SIZE = 4
NUM_EPOCHS = 100
ENCODER_LR = 1e-4
DECODER_LR = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 7

# --- Output ---
CHECKPOINT_DIR = Path('/content/Master/code/checkpoints')
CHECKPOINT_DIR.mkdir(exist_ok=True)


## 2. Data Loading

In [ ]:
npz_files = sorted(PATCH_CACHE.glob('*.npz'))
print(f"Found {len(npz_files)} cached patch files in {PATCH_CACHE}")

all_patches, all_masks = [], []
file_boundaries = []

for npz_path in npz_files:
    with np.load(npz_path) as data:
        patches = list(data['patches'])
        masks = list(data['masks'])
    file_boundaries.append(len(patches))
    all_patches.extend(patches)
    all_masks.extend(masks)
    print(f"  {npz_path.name}: {len(patches)} patches")

print(f"\nTotal: {len(all_patches)} patches from {len(npz_files)} files")

# File-level train/val split
n_files = len(npz_files)
rng = np.random.RandomState(42)
indices = rng.permutation(n_files)
n_val = max(1, int(n_files * 0.2))
val_indices = set(indices[:n_val])

train_patches, train_masks = [], []
val_patches, val_masks = [], []
offset = 0
for i, count in enumerate(file_boundaries):
    p = all_patches[offset:offset + count]
    m = all_masks[offset:offset + count]
    if i in val_indices:
        val_patches.extend(p)
        val_masks.extend(m)
    else:
        train_patches.extend(p)
        train_masks.extend(m)
    offset += count

transform = get_augmentation_pipeline()
train_ds = EchogramPatchDataset(train_patches, train_masks,
                                target_size=TARGET_SIZE, transform=transform)
val_ds = EchogramPatchDataset(val_patches, val_masks,
                              target_size=TARGET_SIZE, transform=None)

print(f'Training patches:   {len(train_ds)}')
print(f'Validation patches: {len(val_ds)}')

# Class balance
bottom_fracs = [m.sum() / m.size for m in train_ds.masks]
print(f'Seabed pixel fraction: {np.mean(bottom_fracs):.1%} '
      f'(min {np.min(bottom_fracs):.1%}, max {np.max(bottom_fracs):.1%})')

## 3. Visualise Training Data

In [ ]:
# Show a few training samples with all 3 channels and the mask
n_show = min(3, len(train_ds))
fig, axes = plt.subplots(n_show, 4, figsize=(16, 4 * n_show))
if n_show == 1:
    axes = axes[np.newaxis]

channel_names = ['38 kHz (norm)', '200 kHz (norm)', 'Difference']
np.random.seed(42)
for i in range(n_show):
    idx = np.random.randint(len(train_ds))
    img, msk = train_ds[idx]
    img_np = img.numpy()
    msk_np = msk.numpy()

    for c in range(3):
        axes[i, c].imshow(img_np[c], aspect='auto', cmap='viridis',
                          vmin=0, vmax=1, origin='upper')
        if i == 0:
            axes[i, c].set_title(channel_names[c])
        axes[i, c].set_ylabel(f'Patch {idx}')

    axes[i, 3].imshow(msk_np, aspect='auto', cmap='gray_r',
                      vmin=0, vmax=1, origin='upper')
    if i == 0:
        axes[i, 3].set_title('Mask')

plt.suptitle(f'Training samples — resized to {TARGET_SIZE}', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Model Setup

In [ ]:
model = create_unet(encoder_name=ENCODER_NAME, in_channels=3, classes=1)

# Apply freezing strategy
if FREEZE_STRATEGY == 'encoder':
    freeze_encoder(model)
    print('Froze entire encoder — training decoder only')
elif FREEZE_STRATEGY == 'stages':
    freeze_encoder_stages(model, freeze_stages=FREEZE_STAGES)
    print(f'Froze first {FREEZE_STAGES} encoder stages')
else:
    print('No freezing — training all parameters with differential LR')

counts = get_trainable_param_count(model)
print(f"Total params:     {counts['total']:>10,}")
print(f"Trainable params: {counts['trainable']:>10,}  ({counts['trainable_pct']:.1f}%)")
print(f"Frozen params:    {counts['frozen']:>10,}")

model = model.to(DEVICE)

In [ ]:
# Smoke test: verify the model accepts the expected input shape
h, w = TARGET_SIZE
dummy = torch.randn(1, 3, h, w).to(DEVICE)
with torch.no_grad():
    out = model(dummy)
print(f'Input:  {dummy.shape}')
print(f'Output: {out.shape}  (expected [1, 1, {h}, {w}])')
assert out.shape == (1, 1, h, w), f'Shape mismatch: {out.shape}'
del dummy, out

In [ ]:
# Optimizer with differential learning rates
param_groups = get_param_groups(model, encoder_lr=ENCODER_LR, decoder_lr=DECODER_LR)
optimizer = torch.optim.AdamW(param_groups, weight_decay=WEIGHT_DECAY)

# Reduce LR when validation loss plateaus
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5)

loss_fn = DiceBCELoss(bce_weight=0.5)

# Mixed precision scaler (GPU only)
# scaler = torch.amp.GradScaler() if DEVICE == 'cuda' else None
scaler = None
print(f'Mixed precision: {"enabled" if scaler else "disabled"}')

In [ ]:
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=(DEVICE != 'cpu'))
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=0, pin_memory=(DEVICE != 'cpu'))

## 5. Training

In [ ]:
history = {'train_loss': [], 'val_loss': [],
           'train_iou': [], 'val_iou': [],
           'train_dice': [], 'val_dice': []}

best_val_iou = 0.0
best_epoch = -1
checkpoint_path = CHECKPOINT_DIR / f'unet_{ENCODER_NAME}_best.pth'
#early_stop = EarlyStopping(patience=PATIENCE)

for epoch in range(NUM_EPOCHS):
    train_metrics = train_one_epoch(model, train_loader, optimizer,
                                    loss_fn, DEVICE, scaler)
    val_metrics = validate(model, val_loader, loss_fn, DEVICE)

    scheduler.step(val_metrics['loss'])

    history['train_loss'].append(train_metrics['loss'])
    history['val_loss'].append(val_metrics['loss'])
    history['train_iou'].append(train_metrics['iou'])
    history['val_iou'].append(val_metrics['iou'])
    history['train_dice'].append(train_metrics['dice'])
    history['val_dice'].append(val_metrics['dice'])

    # Save best model
    if val_metrics['iou'] > best_val_iou:
        best_val_iou = val_metrics['iou']
        best_epoch = epoch
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_iou': best_val_iou,
            'config': {
                'encoder_name': ENCODER_NAME,
                'target_size': TARGET_SIZE,
                'freeze_strategy': FREEZE_STRATEGY,
            }
        }, checkpoint_path)

    print(f'Epoch {epoch+1:3d}/{NUM_EPOCHS}  '
          f'train_loss={train_metrics["loss"]:.4f}  '
          f'val_loss={val_metrics["loss"]:.4f}  '
          f'val_IoU={val_metrics["iou"]:.4f}  '
          f'val_Dice={val_metr
                      ics["dice"]:.4f}'
          f'{"  *" if epoch == best_epoch else ""}')

    # if early_stop(val_metrics['iou']):
    #     print(f'\nEarly stopping: val IoU did not improve for {PATIENCE} epochs.')
    #     break

print(f'\nBest val IoU: {best_val_iou:.4f} at epoch {best_epoch + 1}')
print(f'Checkpoint saved to {checkpoint_path}')

In [ ]:
#early_stop = EarlyStopping(patience=PATIENCE)

for epoch in range(100):
    train_metrics = train_one_epoch(model, train_loader, optimizer,
                                    loss_fn, DEVICE, scaler)
    val_metrics = validate(model, val_loader, loss_fn, DEVICE)

    scheduler.step(val_metrics['loss'])

    history['train_loss'].append(train_metrics['loss'])
    history['val_loss'].append(val_metrics['loss'])
    history['train_iou'].append(train_metrics['iou'])
    history['val_iou'].append(val_metrics['iou'])
    history['train_dice'].append(train_metrics['dice'])
    history['val_dice'].append(val_metrics['dice'])

    # Save best model
    if val_metrics['iou'] > best_val_iou:
        best_val_iou = val_metrics['iou']
        best_epoch = epoch
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_iou': best_val_iou,
            'config': {
                'encoder_name': ENCODER_NAME,
                'target_size': TARGET_SIZE,
                'freeze_strategy': FREEZE_STRATEGY,
            }
        }, checkpoint_path)

    print(f'Epoch {epoch+1:3d}/{NUM_EPOCHS}  '
          f'train_loss={train_metrics["loss"]:.4f}  '
          f'val_loss={val_metrics["loss"]:.4f}  '
          f'val_IoU={val_metrics["iou"]:.4f}  '
          f'val_Dice={val_metrics["dice"]:.4f}'
          f'{"  *" if epoch == best_epoch else ""}')

    if early_stop(val_metrics['iou']):
        print(f'\nEarly stopping: val IoU did not improve for {PATIENCE} epochs.')
        break

print(f'\nBest val IoU: {best_val_iou:.4f} at epoch {best_epoch + 1}')
print(f'Checkpoint saved to {checkpoint_path}')

## 6. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
epochs = range(1, len(history['train_loss']) + 1)

axes[0].plot(epochs, history['train_loss'], label='Train')
axes[0].plot(epochs, history['val_loss'], label='Val')
axes[0].set_title('Loss (Dice + BCE)')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(epochs, history['train_iou'], label='Train')
axes[1].plot(epochs, history['val_iou'], label='Val')
axes[1].set_title('IoU')
axes[1].set_xlabel('Epoch')
axes[1].legend()

axes[2].plot(epochs, history['train_dice'], label='Train')
axes[2].plot(epochs, history['val_dice'], label='Val')
axes[2].set_title('Dice')
axes[2].set_xlabel('Epoch')
axes[2].legend()

plt.suptitle(f'{ENCODER_NAME} — freeze={FREEZE_STRATEGY} — {TARGET_SIZE}',
             fontsize=13)
plt.tight_layout()
plt.show()

## 7. Qualitative Results

In [ ]:
# Load best checkpoint
ckpt = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
print(f'Loaded checkpoint from epoch {ckpt["epoch"] + 1} '
      f'(val IoU = {ckpt["val_iou"]:.4f})')

In [ ]:
# Predict on validation patches
n_show = min(4, len(val_ds))
fig, axes = plt.subplots(n_show, 4, figsize=(16, 4 * n_show))
if n_show == 1:
    axes = axes[np.newaxis]

model.eval()
for i in range(n_show):
    random_echogram_index = np.random.randint(len(val_ds))  # random sample index
    img, msk = val_ds[random_echogram_index]
    with torch.no_grad():
        logits = model(img.unsqueeze(0).to(DEVICE))
        pred = torch.sigmoid(logits).squeeze().cpu().numpy()

    img_np = img.numpy()
    msk_np = msk.numpy()

    # Channel 0 (freq1) as background
    axes[i, 0].imshow(img_np[0], aspect='auto', cmap='viridis',
                      vmin=0, vmax=1, origin='upper')
    axes[i, 0].set_title('Echogram (freq1)' if i == 0 else '')

    axes[i, 1].imshow(msk_np, aspect='auto', cmap='gray_r',
                      vmin=0, vmax=1, origin='upper')
    axes[i, 1].set_title('Ground truth' if i == 0 else '')

    axes[i, 2].imshow(pred, aspect='auto', cmap='gray_r',
                      vmin=0, vmax=1, origin='upper')
    axes[i, 2].set_title('Prediction (prob)' if i == 0 else '')

    # Overlay: echogram + predicted boundary
    axes[i, 3].imshow(img_np[0], aspect='auto', cmap='viridis',
                      vmin=0, vmax=1, origin='upper')
    axes[i, 3].contour(pred, levels=[0.5], colors='red', linewidths=1.5)
    axes[i, 3].set_title('Overlay' if i == 0 else '')

    axes[i, 0].set_ylabel(f'Val {i}')

plt.suptitle('Validation predictions', fontsize=13)
plt.tight_layout()
plt.show()

## 8. Full-Echogram Prediction

Run the trained model across an entire echogram using sliding-window
inference with overlap averaging for a seamless prediction.

In [ ]:
# Reconstruct a full echogram from cached patches for qualitative evaluation.
# Pick the first npz file and tile its patches back into a wide strip.
npz_path = sorted(PATCH_CACHE.glob('*.npz'))[0]
with np.load(npz_path) as data:
    full_patches = data['patches']   # (N, 3, H, W)
    full_masks   = data['masks']      # (N, H, W)

# Stack patches side-by-side to form a pseudo-echogram
sv_channels = np.concatenate(full_patches, axis=-1)  # (3, H, W_total)
ground_truth = np.concatenate(full_masks, axis=-1)    # (H, W_total)
print(f'Reassembled echogram: {sv_channels.shape}, mask: {ground_truth.shape}')


In [ ]:
# Run sliding-window prediction over the reassembled echogram
from deep_learning.unet_model import predict_full_echogram

prediction = predict_full_echogram(
    model, sv_channels, target_size=TARGET_SIZE,
    patch_width=PATCH_WIDTH, stride=STRIDE, device=DEVICE,
)

fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True, sharey=True)

axes[0].imshow(sv_channels[0], aspect='auto', cmap='viridis', vmin=0, vmax=1,
               origin='upper')
axes[0].set_title(f'Echogram ({FREQ1_HZ/1e3:.0f} kHz, normalised)')
axes[0].set_ylabel('Sample index')

axes[1].imshow(prediction, aspect='auto', cmap='gray_r', vmin=0, vmax=1,
               origin='upper')
axes[1].set_title('U-Net prediction (probability)')
axes[1].set_ylabel('Sample index')

axes[2].imshow(sv_channels[0], aspect='auto', cmap='viridis', vmin=0, vmax=1,
               origin='upper')
axes[2].contour(prediction, levels=[0.5], colors='red', linewidths=1.0,
                origin='upper')
axes[2].set_title('Echogram with predicted boundary')
axes[2].set_ylabel('Sample index')
axes[2].set_xlabel('Ping')

plt.suptitle(f'Full echogram — {ENCODER_NAME}, freeze={FREEZE_STRATEGY}',
             fontsize=13)
plt.tight_layout()
plt.show()


## 9. Freezing Strategy Comparison

Train the same architecture with all three freezing strategies on the
same data split and compare convergence speed and final IoU.

In [ ]:
# Set to True to run the comparison (takes 3x longer)
RUN_COMPARISON = False

if RUN_COMPARISON:
    strategies = {
        'Freeze encoder': lambda m: freeze_encoder(m),
        'Freeze 3 stages': lambda m: freeze_encoder_stages(m, freeze_stages=3),
        'Differential LR': lambda m: m,  # no freezing
    }

    comparison = {}
    COMP_EPOCHS = 20

    for name, freeze_fn in strategies.items():
        print(f'\n--- {name} ---')
        m = create_unet(encoder_name=ENCODER_NAME, in_channels=3, classes=1)
        freeze_fn(m)
        m = m.to(DEVICE)

        counts = get_trainable_param_count(m)
        print(f'  Trainable: {counts["trainable"]:,} ({counts["trainable_pct"]:.1f}%)')

        pg = get_param_groups(m, encoder_lr=ENCODER_LR, decoder_lr=DECODER_LR)
        opt = torch.optim.AdamW(pg, weight_decay=WEIGHT_DECAY)
        loss = DiceBCELoss(bce_weight=0.5)

        hist = {'val_iou': [], 'val_loss': []}
        for epoch in range(COMP_EPOCHS):
            train_one_epoch(m, train_loader, opt, loss, DEVICE)
            vm = validate(m, val_loader, loss, DEVICE)
            hist['val_iou'].append(vm['iou'])
            hist['val_loss'].append(vm['loss'])
            if (epoch + 1) % 5 == 0:
                print(f'  Epoch {epoch+1:3d}  val_IoU={vm["iou"]:.4f}')

        comparison[name] = hist
        del m, opt
        torch.cuda.empty_cache() if DEVICE == 'cuda' else None

    # Plot comparison
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for name, hist in comparison.items():
        axes[0].plot(range(1, COMP_EPOCHS + 1), hist['val_loss'], label=name)
        axes[1].plot(range(1, COMP_EPOCHS + 1), hist['val_iou'], label=name)

    axes[0].set_title('Validation Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].legend()

    axes[1].set_title('Validation IoU')
    axes[1].set_xlabel('Epoch')
    axes[1].legend()

    plt.suptitle('Freezing strategy comparison', fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print('Set RUN_COMPARISON = True to run the freezing strategy comparison.')